# ultrasonic รอบห้า (r5dev) — เลือกสูตรด้วยห้องที่ไม่เคยเห็น

**ก่อนเริ่ม**
1. **Runtime → Change runtime type → T4 GPU** → Save
2. อัปโหลดสองไฟล์ไปที่ Google Drive โฟลเดอร์ **`MyDrive/ultrasonic_r4/`** (โฟลเดอร์เดิม)
   - `ultrasonic_r5dev_data.zip` (ข้อมูล)
   - `ultrasonic_code.zip` (โค้ด · ทับของเดิมได้)
3. รันทีละเซลล์จากบนลงล่าง

การแบ่ง: เทรน **ห้องเดิม (hw2) + ห้อง B** · เลือกรอบ/จุดตัด/สูตร ด้วย **ห้อง C ทั้งห้อง**
· **ห้อง D ไม่อยู่ในชุดนี้** (ข้อสอบสุดท้าย ใช้รอบหน้าครั้งเดียว)

## 1. ตรวจ GPU

In [ ]:
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "ไม่มี — Runtime > Change runtime type > T4 GPU")

## 2. เชื่อม Drive แล้วแตกไฟล์

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, zipfile
D = '/content/drive/MyDrive/ultrasonic_r4'
RUNS = f'{D}/runs'
WORK = '/content/r5'
for z in ('ultrasonic_r5dev_data.zip', 'ultrasonic_code.zip'):
    assert os.path.exists(f'{D}/{z}'), f'ไม่พบ {D}/{z} — อัปโหลดก่อน'
os.makedirs(RUNS, exist_ok=True)
if os.path.exists(WORK):
    shutil.rmtree(WORK)
for z in ('ultrasonic_r5dev_data.zip', 'ultrasonic_code.zip'):
    zipfile.ZipFile(f'{D}/{z}').extractall(WORK)
os.symlink(RUNS, f'{WORK}/car/data/r4_runs')
os.chdir(f'{WORK}/car')
os.environ['MAP2_ROUND'] = 'r5dev'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print('พร้อม')

## 3. รันคิว (~1-1.5 ชั่วโมงบน T4)

ทุกการทดลองใช้การแบ่งเดียวกัน (`SPLIT`) เปลี่ยนทีละอย่าง ดูเหตุผลใน `car/ROUND4.md` หัวข้อ 7

In [ ]:
import subprocess, sys, time
SPLIT = ['--train', 'hw2_*,roomB_*', '--val', 'roomC_*', '--test', '']
QUEUE = [
    ('r5_base_s0', []),
    ('r5_base_s1', ['--seed', '1']),
    ('r5_l1_s0', ['--loss', 'bce_l1']),
    ('r5_l1_s1', ['--loss', 'bce_l1', '--seed', '1']),
    ('r5_mse_s0', ['--loss', 'mse']),
    ('r5_bdh_w2', ['--loss', 'bdh']),
    ('r5_bdh_w4', ['--loss', 'bdh', '--w-depth', '4']),
    ('r5_l1_slice5_s0', ['--loss', 'bce_l1', '--slice-ms', '5']),
    ('r5_l1_slice5_s1', ['--loss', 'bce_l1', '--slice-ms', '5', '--seed', '1']),
    ('r5_l1_slice3', ['--loss', 'bce_l1', '--slice-ms', '3']),
    ('r5_l1_slice8', ['--loss', 'bce_l1', '--slice-ms', '8']),
    ('r5_l1_leaky', ['--loss', 'bce_l1', '--act', 'leaky']),
]
for name, extra in QUEUE:
    if os.path.exists(f'{RUNS}/{name}.json'):
        print('ข้าม (เสร็จแล้ว):', name); continue
    cmd = [sys.executable, 'train_r4.py', '--name', name, '--epochs', '12', '--threads', '2'] + SPLIT + extra
    print('=' * 70, '\nเริ่ม', name, ' '.join(extra), flush=True)
    t = time.time()
    with open(f'{RUNS}/{name}.log', 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True); log.write(line)
        p.wait()
    print(f"{'เสร็จ' if p.returncode == 0 else 'ล้ม'} {name} · {(time.time() - t) / 60:.0f} นาที")

## 4. สรุปผล

In [ ]:
!python r4_report.py 'r5_*.json'

ส่งตารางนี้ให้ Claude หรือบอกว่าเสร็จแล้ว (Claude อ่านจากลิงก์โฟลเดอร์ Drive ที่แชร์ได้)